# Module 05 AI Workbook — solution and explanation

> Try the exercise yourself before reading this.

## The adversarial bug

[../adversarial_weighted_sum_buggy.py](./adversarial_weighted_sum_buggy.py)
accumulates in single precision:

```python
total = np.float32(0.0)
for w in w32:
    total = np.float32(total + w)     # float32 running total
```

`float32` has ~7 significant decimal digits. When the running total grows to,
say, ~1000 while each addend is ~0.0005, the addend is near the precision limit
of the sum, so its low-order bits are dropped every step. Over two million events
the dropped bits accumulate into a visible error.

This is the same trap as a **GPU reduction that accumulates in `float32`** (the
GPU default). The hardware is fast at single precision, so an AI port often uses
it silently.

## The fix

Accumulate in `float64`:

```python
return float(np.sum(weights, dtype=np.float64))
```

NumPy's `np.sum` also uses **pairwise summation**, which keeps the rounding error
tiny. When you must stay in a lower precision (e.g. a `float32` GPU reduction for
speed), use **Kahan compensated summation** or a **pairwise/tree reduction** —
both are shown in [weighted_sum_solution.py](solutions/weighted_sum_solution.py). Those are
exactly the techniques a good CUDA reduction uses.

## Why the verification matters

The buggy result is *close* — a casual "looks about right" passes. The reliable
test is a **relative-error** comparison against a high-precision reference
(`math.fsum`, which is exact for float64 inputs), which is what
[../verify_weighted_sum.py](./verify_weighted_sum.py) does.

## Mapping to the 5-phase loop

- **PREDICT:** flag the `float32` accumulation and estimate the error growth.
- **VERIFY:** relative error vs `fsum`, not eyeballing the number.
- **DIAGNOSE:** name precision, not speed, as the issue; the fix (float64 /
  Kahan / pairwise) is the same reasoning you apply to GPU reductions.


## Run the worked solution

The cells below build and run the solution. Each should finish with `PASS`.

In [ ]:
!python solutions/weighted_sum_solution.py